# 150 — Registro y promoción champion-challenger

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Registro de modelos**: fuente de verdad de versiones con estados (`registrada →
challenger → champion → archivada`), linaje al run de origen, firma de entrada/salida y
aprobador por transición.

**Champion-challenger**: el titular (champion) sirve tráfico; el retador (challenger) se
evalúa con los mismos datos, misma ventana y mismas métricas — típicamente en *shadow
mode* (predice en paralelo sin afectar usuarios).

**Regla de promoción** = métrica primaria con mejora mínima `delta_min` + **guardas**
que no pueden degradarse (latencia p95, calibración, segmentos sensibles) + misma
ventana/población + rollback preparado. La regla se escribe **antes** de ver los números.


## 🧮 Ejemplo de referencia

Regla: promover si ΔAUC ≥ 0.005; guardas: FN del segmento joven ≤ +1.0 pp, p95 ≤ 150 ms.

```text
                     v7 (champ)   v8 (chall)   Δ         regla
AUC                  0.831        0.842        +0.011    ✓
FN segmento joven    8.2 %        9.9 %        +1.7 pp   ✗  ← guarda violada
p95                  110 ms       128 ms       +18 ms    ✓
```

Decisión: **no promover** pese a la mejora clara de la métrica primaria: las guardas
existen exactamente para este caso (mejorar el promedio degradando un segmento).


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("workflow", seed=150)
show(result)


## Reflexión

1. ¿Por qué `delta_min > 0` en la regla de promoción, si cualquier mejora del AUC parece deseable? ¿Qué costos ignora un delta_min = 0?
2. En el ejemplo, v8 mejora el AUC global pero empeora un segmento: ¿qué mecanismo de la regla lo detectó y qué habría pasado evaluando solo la métrica primaria?
3. ¿Qué preguntas puede responder el shadow mode y cuáles solo puede responder un canario con tráfico real? Da un ejemplo de cada una.
